# S8 — Eksperimen revisi IJIES (Paper ID 20266848)

Notebook ini menjalankan eksperimen tambahan yang diminta reviewer. Bagian 0–5 sama dengan notebook S7 (data, pipeline, search space).
Eksperimen revisi dibagi menjadi **blok A–H yang berdiri sendiri**; setiap blok punya checkpoint di `MyDrive/S8_revision/`, jadi bisa dijalankan di sesi Colab berbeda dan dilanjutkan bila terputus.

| Blok | Isi (komentar reviewer) | Perkiraan waktu |
|---|---|---|
| B | Pre-split leak dengan **test set asli** + hitungan asli/sintetis (R1.1, R2.3) | ±2,8 jam |
| D | Cervical **tanpa variabel skrining** (R2.2) | ±1,2 jam |
| F | **Fungsi objektif** MCC & balanced accuracy pada Cervical (R2.4) | ±1,3 jam |
| E | **Imbalance terkontrol** pada Breast: IR 5, 10, 15 (R2.1) | ±1 jam |
| G | **SMOTENC & RandomOverSampler** pada Lung (R1.6) | ±1,4 jam |
| H | **TabPFN** sebagai default modern (R2.5) | ±0,3 jam (GPU) |
| C | **Search space & budget disamakan**, default ikut dievaluasi (R1.3) | ±3,3 jam |
| A | Ulang eksperimen utama **dengan log lengkap** (hyperparameter, prediksi, indeks) untuk repo (R1.7) | ±2,8 jam |

**Cara pakai:** jalankan Bagian 0–5 sekali per sesi, lalu jalankan sel blok yang diinginkan. Urutan prioritas: B → D → F → E → G → H → C → A.
Setelah semua selesai (atau sebagian), jalankan sel **Z** untuk membuat ZIP lalu kirim ZIP-nya.

## Bagian 0–5: setup (sama dengan S7)

In [ ]:
# ============================================================
# 0.1 INSTALASI LIBRARY
# ============================================================
!pip install xgboost lightgbm scikit-optimize openpyxl shap imbalanced-learn scipy -q
print("✅ Instalasi selesai")

In [ ]:
# ============================================================
# 0.2 IMPORT SEMUA LIBRARY
# ============================================================
import warnings, time, os, random, copy
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.compose import ColumnTransformer

# Model selection
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    GridSearchCV, RandomizedSearchCV, cross_validate
)

# Classifiers
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Metrics
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score, recall_score,
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay, accuracy_score
)

# SMOTE — dibungkus Pipeline agar tidak ada data leakage
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Bayesian Optimization
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical

# SHAP
import shap

# Statistical tests
from scipy.stats import wilcoxon, friedmanchisquare

import joblib

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("✅ Semua library berhasil diimport")

In [ ]:
# ============================================================
# 0.3 KONFIGURASI UTAMA — SESUAIKAN DI SINI
# ============================================================

# [MODE] 'test' = cepat (pakai N_SAMPLE baris per dataset)
#        'full' = semua data
MODE     = 'full'     # ← ganti ke 'test' untuk prototyping cepat
N_SAMPLE = 500        # ← berlaku hanya saat MODE='test'

# Jumlah runs untuk stabilitas statistik (N≥10 untuk Wilcoxon)
N_RUNS   = 10

# K-Fold CV
CV_FOLDS = 5

# n_iter untuk RandomizedSearchCV & BayesSearchCV
N_ITER_TUNING = 10 if MODE == 'test' else 20

# Output directory
OUTPUT_DIR = '/content/S8_out'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def save_out(fname):
    return os.path.join(OUTPUT_DIR, fname)

print(f"⚙️  MODE          = '{MODE}'")
print(f"⚙️  N_RUNS        = {N_RUNS}")
print(f"⚙️  CV_FOLDS      = {CV_FOLDS}")
print(f"⚙️  N_ITER_TUNING = {N_ITER_TUNING}")
print(f"📁 OUTPUT_DIR     = '{OUTPUT_DIR}'")

In [ ]:
# ============================================================
# 0.3b CATAT LINGKUNGAN EKSEKUSI (untuk Tabel "Experimental settings")
# ============================================================
import sys, platform, subprocess, sklearn, imblearn, xgboost, lightgbm, skopt
env_lines = [f"Python {sys.version.split()[0]}",
             f"CPU: {platform.machine()}, {os.cpu_count()} vCPU"]
try:
    mem_kb = int(open('/proc/meminfo').readline().split()[1])
    env_lines.append(f"RAM: {mem_kb/1024/1024:.1f} GB")
except Exception:
    pass
try:
    cpu_model = subprocess.run("lscpu | grep 'Model name'", shell=True,
                               capture_output=True, text=True).stdout.strip()
    if cpu_model: env_lines.append(cpu_model)
except Exception:
    pass
for m in [sklearn, imblearn, xgboost, lightgbm, skopt, shap]:
    env_lines.append(f"{m.__name__} {m.__version__}")
print("\n".join(env_lines))
with open('/content/env_versions.txt', 'w') as fh:
    fh.write("\n".join(env_lines))
print("\n✅ Disimpan: env_versions.txt")

In [ ]:
# ============================================================
# 1.1 DEFINISI DATASET (konfigurasi terpusat)
# ============================================================
# Sesuaikan path ke lokasi file CSV Anda.
# Notebook otomatis mencoba Google Drive, lalu path lokal.

DATASET_CONFIG = {
    'Lung Cancer': {
        'gdrive_path' : '/content/drive/MyDrive/Colab Notebooks/survey lung cancer.csv',
        'local_path'  : 'survey_lung_cancer.csv',
        'target'      : 'LUNG_CANCER',
        'sep'         : ',',
    },
    'BreastCancer': {
        'gdrive_path' : '/content/drive/MyDrive/Colab Notebooks/Breast Cancer Winconsin.csv',
        'local_path'  : 'breast_cancer_wisconsin.csv',
        'target'      : 'diagnosis',
        'sep'         : ',',
    },
    'CervicalCancer': {
        'gdrive_path' : '/content/drive/MyDrive/Colab Notebooks/risk_factors_cervical_cancer.csv',
        'local_path'  : 'cervical_cancer.csv',
        'target'      : 'Biopsy',
        'sep'         : ',',
    },
}

MODEL_NAMES  = ['XGBoost', 'RandomForest', 'SVM', 'LightGBM']
TUNER_NAMES  = ['No-Tuning', 'Grid', 'Random', 'Bayes']

print("✅ Konfigurasi dataset:")
for ds, cfg in DATASET_CONFIG.items():
    print(f"   • {ds:<20} target='{cfg['target']}'")

In [ ]:
# ============================================================
# 1.2 AUTO-LOAD DATASET (Google Drive → lokal → upload manual)
# ============================================================
try:
    from google.colab import drive, files as colab_files
    drive.mount('/content/drive', force_remount=False)
    IN_COLAB = True
    print("🔵 Google Colab terdeteksi — Drive terhubung")
except ImportError:
    IN_COLAB = False
    print("🟡 Bukan Colab — mode lokal/Kaggle")

def resolve_path(cfg):
    if IN_COLAB and os.path.exists(cfg['gdrive_path']):
        return cfg['gdrive_path']
    if os.path.exists(cfg['local_path']):
        return cfg['local_path']
    return None

print()
for ds, cfg in DATASET_CONFIG.items():
    p = resolve_path(cfg)
    if p:
        print(f"   ✅ {ds:<20} → {p}")
    else:
        print(f"   ⚠️  {ds:<20} → FILE TIDAK DITEMUKAN (update path di 1.1)")

In [ ]:
# ============================================================
# 1.3 LOAD + SAMPLING SEMUA DATASET
# ============================================================
datasets_raw = {}
RAW_ROW = {}

for ds_name, cfg in DATASET_CONFIG.items():
    path = resolve_path(cfg)
    if path is None:
        print(f"⚠️  {ds_name}: skip (file tidak ditemukan)")
        continue
    try:
        df = pd.read_csv(path, sep=cfg['sep'], na_values=['?','NA','null',''])
        n_raw = len(df)
        df = df.drop_duplicates()                       # keep='first'
        RAW_ROW[ds_name] = df.index.values.copy()       # nomor baris asli (0-based, tanpa header) setelah hapus duplikat
        print(f'   {ds_name}: {n_raw} baris asli, {n_raw - len(df)} duplikat dihapus')
        df = df.reset_index(drop=True)

        if MODE == 'test':
            n_take = min(N_SAMPLE, len(df))
            df = df.sample(n=n_take, random_state=SEED).reset_index(drop=True)

        datasets_raw[ds_name] = df
        tgt = cfg['target']
        vc  = df[tgt].value_counts()
        print(f"✅ {ds_name:<20} shape={df.shape}  target='{tgt}'")
        print(f"   Distribusi: {dict(vc)}")
    except Exception as e:
        print(f"❌ {ds_name}: Error — {e}")

print(f"\n📦 Dataset berhasil dimuat: {list(datasets_raw.keys())}")

In [ ]:
# ============================================================
# 3.1 FUNGSI PREPROCESSING (BENAR: fit hanya pada X_train)
# ============================================================
# METODOLOGI KRITIS:
#   preprocess_data() HANYA memisahkan X dan y + encode target.
#   Scaling & imputation dilakukan di dalam imblearn.Pipeline
#   agar hanya fit pada training fold — mencegah data leakage.

def encode_target(y_series):
    """Encode target ke 0/1. Return (y_encoded, label_mapping)."""
    if not pd.api.types.is_numeric_dtype(y_series):   # aman untuk pandas 2 dan 3
        le = LabelEncoder()
        y_enc = le.fit_transform(y_series)
        mapping = dict(zip(le.classes_, le.transform(le.classes_)))
        return y_enc, mapping
    return y_series.values, None

def prepare_xy(df, target_col):
    """
    Pisahkan X dan y. Encode target jika perlu.
    LabelEncoder pada fitur kategorikal di-fit di dalam Pipeline per fold.
    """
    if 'id' in df.columns:
        df = df.drop(columns=['id'])
    X = df.drop(columns=[target_col]).copy()
    y, mapping = encode_target(df[target_col])
    return X, y, mapping

def get_feature_names(preprocessor, num_cols, cat_cols):
    """Ambil nama fitur setelah transformasi ColumnTransformer."""
    feat_names = list(num_cols)
    if cat_cols and 'cat' in preprocessor.named_transformers_:
        ohe = preprocessor.named_transformers_['cat']['ohe']
        feat_names += list(ohe.get_feature_names_out(cat_cols))
    return feat_names

def build_preprocessor(X_df):
    """
    Buat ColumnTransformer (belum di-fit).
    Fitting dilakukan di dalam Pipeline per fold — BUKAN di sini.
    """
    num_cols = X_df.select_dtypes(include='number').columns.tolist()
    cat_cols = X_df.select_dtypes(include=['object', 'string', 'category']).columns.tolist()  # aman untuk pandas 3

    num_pipe = SkPipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('scl', StandardScaler()),
    ])
    cat_pipe = SkPipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])

    steps = []
    if num_cols: steps.append(('num', num_pipe, num_cols))
    if cat_cols: steps.append(('cat', cat_pipe, cat_cols))

    pre = ColumnTransformer(steps, remainder='drop')
    return pre, num_cols, cat_cols

print("✅ Fungsi preprocessing (no-leakage) siap")

In [ ]:
# ============================================================
# 3.2 SIAPKAN X, y UNTUK SEMUA DATASET
# ============================================================
datasets_xy = {}

for ds_name, df in datasets_raw.items():
    tgt = DATASET_CONFIG[ds_name]['target']
    X, y, mapping = prepare_xy(df, tgt)
    pre, num_cols, cat_cols = build_preprocessor(X)

    datasets_xy[ds_name] = {
        'X': X, 'y': y,
        'target': tgt,
        'mapping': mapping,
        'preprocessor': pre,
        'num_cols': num_cols,
        'cat_cols': cat_cols,
    }

    classes, counts = np.unique(y, return_counts=True)
    print(f"✅ {ds_name:<20} X={X.shape}  y={dict(zip(classes.tolist(),counts.tolist()))}")
    if mapping:
        print(f"   Mapping target: {mapping}")

In [ ]:
# ============================================================
# 3.3 LABEL KELAS MINORITAS PER DATASET (untuk recall minoritas)
# ============================================================
# Ditentukan dari distribusi kelas ASLI (bukan dari test set), supaya tetap benar
# meskipun test set sudah di-SMOTE pada protokol bocor (Bagian 11).
MINORITY = {}
for ds_name, data in datasets_xy.items():
    labels, counts = np.unique(data['y'], return_counts=True)
    MINORITY[ds_name] = labels[np.argmin(counts)]
    print(f"   {ds_name:<20} minoritas = kelas {MINORITY[ds_name]} "
          f"(n={counts.min()}, mapping={data['mapping']})")

In [ ]:
# ============================================================
# 4.1 SPLIT 80:20 STRATIFIED (PER DATASET)
# ============================================================
datasets_split = {}

for ds_name, data in datasets_xy.items():
    X, y = data['X'], data['y']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )

    datasets_split[ds_name] = {
        **data,
        'X_train': X_train, 'X_test': X_test,
        'y_train': y_train, 'y_test': y_test,
    }

    print(f"✅ {ds_name:<20} Train={len(y_train):,}  Test={len(y_test):,}")
    for split, yy in [('Train', y_train), ('Test', y_test)]:
        u, c = np.unique(yy, return_counts=True)
        dist = ' | '.join([f"cls{v}={cnt}({cnt/len(yy)*100:.1f}%)" for v,cnt in zip(u,c)])
        print(f"   {split}: {dist}")

In [ ]:
# ============================================================
# 4.2 FACTORY: imblearn.Pipeline (Preprocessor + SMOTE + Model)
# ============================================================
# METODOLOGI KRITIS:
#   SMOTE dibungkus dalam ImbPipeline.
#   Saat cross_validate dipanggil:
#     → Per fold: Pipeline fit pada training fold saja
#     → SMOTE hanya dijalankan pada training fold
#     → Validation fold TIDAK pernah di-SMOTE
#   → Tidak ada data leakage synthetic samples ✅

def make_pipeline(preprocessor, model, smote_k=5):
    """Buat ImbPipeline: preprocessor → SMOTE → classifier."""
    return ImbPipeline([
        ('pre',   copy.deepcopy(preprocessor)),
        ('smote', SMOTE(random_state=SEED, k_neighbors=smote_k)),
        ('clf',   copy.deepcopy(model)),
    ])

def build_model(model_name, seed=SEED):
    if model_name == 'XGBoost':
        return XGBClassifier(objective='binary:logistic', eval_metric='logloss',
                             random_state=seed, use_label_encoder=False, verbosity=0)
    elif model_name == 'RandomForest':
        return RandomForestClassifier(random_state=seed, n_jobs=-1)
    elif model_name == 'SVM':
        return SVC(probability=True, random_state=seed)
    elif model_name == 'LightGBM':
        return LGBMClassifier(random_state=seed, verbose=-1)
    raise ValueError(f'Unknown model: {model_name}')

print("✅ Pipeline factory siap")
print("   Flow: X_raw → Preprocessor (fit train only) → SMOTE (train fold only) → Classifier")

In [ ]:
# ============================================================
# 4.3 SEARCH SPACES UNTUK SEMUA TUNER
# ============================================================
# Budget evaluasi diseimbangkan: Grid ≈ Random ≈ Bayes (≈20 kombinasi)

PARAM_GRID = {
    'XGBoost'     : {'clf__learning_rate':[0.01,0.05,0.1],
                     'clf__max_depth':[3,5,7],
                     'clf__n_estimators':[100,200],
                     'clf__subsample':[0.8,1.0]},
    'RandomForest': {'clf__n_estimators':[100,200],
                     'clf__max_depth':[None,5,10],
                     'clf__min_samples_split':[2,5],
                     'clf__max_features':['sqrt','log2']},
    'SVM'         : {'clf__C':[0.1,1,10],
                     'clf__kernel':['rbf','linear'],
                     'clf__gamma':['scale','auto']},
    'LightGBM'    : {'clf__learning_rate':[0.01,0.05,0.1],
                     'clf__n_estimators':[100,200],
                     'clf__max_depth':[3,5,7],
                     'clf__num_leaves':[31,63]},
}

PARAM_RANDOM = {
    'XGBoost'     : {'clf__learning_rate':[0.01,0.05,0.1,0.2],
                     'clf__max_depth':[3,5,7,9],
                     'clf__n_estimators':[50,100,200,300],
                     'clf__subsample':[0.6,0.8,1.0]},
    'RandomForest': {'clf__n_estimators':[50,100,200,300],
                     'clf__max_depth':[None,5,10,20],
                     'clf__min_samples_split':[2,5,10],
                     'clf__max_features':['sqrt','log2']},
    'SVM'         : {'clf__C':[0.01,0.1,1,10,100],
                     'clf__kernel':['rbf','linear'],
                     'clf__gamma':['scale','auto']},
    'LightGBM'    : {'clf__learning_rate':[0.01,0.05,0.1,0.2],
                     'clf__n_estimators':[50,100,200,300],
                     'clf__max_depth':[3,5,7,9],
                     'clf__num_leaves':[15,31,63,127]},
}

PARAM_BAYES = {
    'XGBoost'     : {'clf__learning_rate':Real(0.01,0.3),
                     'clf__max_depth':Integer(3,10),
                     'clf__n_estimators':Integer(50,300),
                     'clf__subsample':Real(0.5,1.0)},
    'RandomForest': {'clf__n_estimators':Integer(50,300),
                     'clf__max_depth':Integer(3,20),
                     'clf__min_samples_split':Integer(2,10),
                     'clf__max_features':Categorical(['sqrt','log2'])},
    'SVM'         : {'clf__C':Real(0.01,100.0,prior='log-uniform'),
                     'clf__kernel':Categorical(['rbf','linear']),
                     'clf__gamma':Categorical(['scale','auto'])},
    'LightGBM'    : {'clf__learning_rate':Real(0.01,0.3),
                     'clf__n_estimators':Integer(50,300),
                     'clf__max_depth':Integer(3,10),
                     'clf__num_leaves':Integer(15,127)},
}

print("✅ Search spaces siap (prefix clf__ untuk Pipeline compatibility)")
for mn in MODEL_NAMES:
    from functools import reduce
    import operator
    g = reduce(operator.mul, [len(v) for v in PARAM_GRID[mn].values()], 1)
    print(f"   {mn:<14} Grid≈{g} kombinasi | Random/Bayes={N_ITER_TUNING} iter (seimbang)")

In [ ]:
def run_no_tuning(pipeline, X_tr, y_tr):
    """Baseline: default hyperparameter, langsung fit."""
    t0 = time.time()
    pipeline.fit(X_tr, y_tr)
    return pipeline, time.time() - t0

def run_grid(pipeline, model_name, X_tr, y_tr, seed):
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=seed)
    gs = GridSearchCV(pipeline, PARAM_GRID[model_name],
                      scoring='f1_weighted', cv=cv, n_jobs=-1, refit=True)
    t0 = time.time(); gs.fit(X_tr, y_tr)
    return gs.best_estimator_, time.time()-t0, gs.cv_results_

def run_random(pipeline, model_name, X_tr, y_tr, seed):
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=seed)
    rs = RandomizedSearchCV(pipeline, PARAM_RANDOM[model_name],
                            n_iter=N_ITER_TUNING, scoring='f1_weighted',
                            cv=cv, random_state=seed, n_jobs=-1, refit=True)
    t0 = time.time(); rs.fit(X_tr, y_tr)
    return rs.best_estimator_, time.time()-t0, rs.cv_results_

def run_bayes(pipeline, model_name, X_tr, y_tr, seed):
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=seed)
    bs = BayesSearchCV(pipeline, PARAM_BAYES[model_name],
                       n_iter=N_ITER_TUNING, scoring='f1_weighted',
                       cv=cv, random_state=seed, n_jobs=-1, refit=True)
    t0 = time.time(); bs.fit(X_tr, y_tr)
    return bs.best_estimator_, time.time()-t0, bs.cv_results_

In [ ]:
# ============================================================
# 5.2 EVALUASI — metrik utama + metrik sensitif minoritas (OPSI 3)
# ============================================================
from sklearn.metrics import matthews_corrcoef, balanced_accuracy_score

METRICS = ['f1', 'auc', 'precision', 'recall', 'accuracy',      # metrik utama (weighted)
           'f1_macro', 'bal_acc', 'mcc', 'recall_min']           # metrik sensitif minoritas

def evaluate_model(model, X_te, y_te, minority=None):
    """Evaluasi pada test set. `minority` = label kelas minoritas dataset asli."""
    y_pred = model.predict(X_te)
    labels, counts = np.unique(y_te, return_counts=True)
    if minority is None:
        minority = labels[np.argmin(counts)]
    try:
        prob = model.predict_proba(X_te)
        auc = (roc_auc_score(y_te, prob, multi_class='ovr', average='weighted')
               if len(labels) > 2 else roc_auc_score(y_te, prob[:, 1]))
    except Exception:
        auc = np.nan
    return {
        'f1'        : f1_score(y_te, y_pred, average='weighted', zero_division=0),
        'auc'       : auc,
        'precision' : precision_score(y_te, y_pred, average='weighted', zero_division=0),
        'recall'    : recall_score(y_te, y_pred, average='weighted', zero_division=0),
        'accuracy'  : accuracy_score(y_te, y_pred),
        'f1_macro'  : f1_score(y_te, y_pred, average='macro', zero_division=0),
        'bal_acc'   : balanced_accuracy_score(y_te, y_pred),
        'mcc'       : matthews_corrcoef(y_te, y_pred),
        'recall_min': recall_score(y_te, y_pred, labels=[minority], average='macro', zero_division=0),
        'y_pred'    : y_pred,
    }

def run_tuner(tuner, pipe, model_name, X_tr, y_tr, seed):
    """Jalankan satu tuner. Return (model, runtime, skor CV terbaik)."""
    if tuner == 'No-Tuning':
        m, t = run_no_tuning(pipe, X_tr, y_tr)
        return m, t, np.nan, None
    fn = {'Grid': run_grid, 'Random': run_random, 'Bayes': run_bayes}[tuner]
    m, t, cvr = fn(pipe, model_name, X_tr, y_tr, seed)
    return m, t, float(np.nanmax(cvr['mean_test_score'])), cvr

print("✅ Fungsi evaluasi siap:", ", ".join(METRICS))

## R0. Infrastruktur revisi: split berpasangan, log artefak, checkpoint

In [ ]:
# ============================================================
# R0. INFRASTRUKTUR REVISI
# ============================================================
import json, warnings
from sklearn.base import clone
from sklearn.metrics import make_scorer, matthews_corrcoef, balanced_accuracy_score, roc_curve
from sklearn.preprocessing import OrdinalEncoder
from imblearn.over_sampling import SMOTENC, RandomOverSampler

REV_DIR = '/content/drive/MyDrive/S8_revision' if os.path.isdir('/content/drive/MyDrive') else os.path.join(OUTPUT_DIR, 'S8_revision')
os.makedirs(REV_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('📁 Folder revisi:', REV_DIR)

def full_xy_ids(ds, data):
    """Data penuh dalam URUTAN YANG SAMA dengan loop utama S7 (train awal + test awal), plus ID baris asli."""
    X_all = pd.DataFrame(np.vstack([data['X_train'], data['X_test']]), columns=data['X'].columns)
    y_all = np.concatenate([data['y_train'], data['y_test']])
    idx = np.concatenate([data['X_train'].index.values, data['X_test'].index.values])
    raw = RAW_ROW[ds][idx] if ds in RAW_ROW else idx
    return X_all, y_all, raw

def split_idx(y_all, run):
    """Indeks train/test identik dengan train_test_split pada loop utama (seed 42+run)."""
    tr, te = train_test_split(np.arange(len(y_all)), test_size=0.2, stratify=y_all, random_state=SEED + run)
    return tr, te

def sens_spec(y, pred, prob, pos=1, target_spec=0.90):
    y = np.asarray(y); pred = np.asarray(pred)
    tp = np.sum((y == pos) & (pred == pos)); fn = np.sum((y == pos) & (pred != pos))
    tn = np.sum((y != pos) & (pred != pos)); fp = np.sum((y != pos) & (pred == pos))
    out = {'sens': tp / max(tp + fn, 1), 'spec': tn / max(tn + fp, 1), 'ppv': tp / max(tp + fp, 1)}
    try:
        fpr, tpr, _ = roc_curve((y == pos).astype(int), prob)
        ok = fpr <= (1 - target_spec) + 1e-12
        out['sens_at_spec90'] = float(tpr[ok].max()) if ok.any() else 0.0
    except Exception:
        out['sens_at_spec90'] = np.nan
    return out

def evaluate_full(model, X_te, y_te, minority):
    res = evaluate_model(model, X_te, y_te, minority=minority)
    pred = res.pop('y_pred')
    try:
        prob = model.predict_proba(X_te)[:, 1]
    except Exception:
        prob = np.full(len(y_te), np.nan)
    res.update(sens_spec(y_te, pred, prob))
    return res, pred, prob

def search(tuner, pipe, Xtr, ytr, seed, grid=None, rand=None, bayes=None, n_iter=None, scoring='f1_weighted'):
    """Satu tuner. Return (model, runtime_s, cv_best, best_params, n_candidates, cv_results)."""
    if tuner == 'No-Tuning':
        t0 = time.time(); pipe.fit(Xtr, ytr); return pipe, time.time() - t0, np.nan, {}, 1, None
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=seed)
    if tuner == 'Grid':
        s = GridSearchCV(pipe, grid, scoring=scoring, cv=cv, n_jobs=-1, refit=True)
    elif tuner == 'Random':
        s = RandomizedSearchCV(pipe, rand, n_iter=n_iter, scoring=scoring, cv=cv, random_state=seed, n_jobs=-1, refit=True)
    elif tuner == 'Bayes':
        s = BayesSearchCV(pipe, bayes, n_iter=n_iter, scoring=scoring, cv=cv, random_state=seed, n_jobs=-1, refit=True)
    t0 = time.time(); s.fit(Xtr, ytr); rt = time.time() - t0
    bp = {k.replace('clf__', ''): (v.item() if hasattr(v, 'item') else v) for k, v in s.best_params_.items()}
    return s.best_estimator_, rt, float(np.nanmax(s.cv_results_['mean_test_score'])), bp, len(s.cv_results_['params']), s.cv_results_

def _save_csv(df, path):
    tmp = path + '.tmp'; df.to_csv(tmp, index=False); os.replace(tmp, path)

def run_block(block, tasks, fit_fn, log_pred=True):
    """tasks: list of dict. fit_fn(task) -> dict(model, runtime, cv_best, params, n_cand, X_te, y_te, test_ids, minority, extra)."""
    ck = os.path.join(REV_DIR, f'{block}_results.csv')
    pp = os.path.join(REV_DIR, f'{block}_predictions.csv')
    hp = os.path.join(REV_DIR, f'{block}_params.jsonl')
    rows, done = [], set()
    if os.path.exists(ck):
        df0 = pd.read_csv(ck); rows = df0.to_dict('records'); done = set(df0['key'])
        print(f'↻ {block}: lanjut dari checkpoint ({len(done)}/{len(tasks)} tugas selesai)')
    t_start = time.time()
    for n, t in enumerate(tasks):
        key = '|'.join(f'{k}={t[k]}' for k in sorted(t))
        if key in done: continue
        o = fit_fn(t)
        res, pred, prob = evaluate_full(o['model'], o['X_te'], o['y_te'], o['minority'])
        row = {'key': key, 'block': block, **t, 'runtime': o['runtime'], 'cv_best': o['cv_best'], 'n_candidates': o['n_cand'], **res, **o.get('extra', {})}
        if log_pred:
            pd.DataFrame({'key': key, 'row_id': o['test_ids'], 'y_true': o['y_te'], 'y_pred': pred, 'p1': prob}).to_csv(
                pp, mode='a', header=not os.path.exists(pp), index=False)
        with open(hp, 'a') as f:
            f.write(json.dumps({'key': key, **{k: str(v) for k, v in t.items()}, 'best_params': {k: str(v) for k, v in o['params'].items()}}) + '\n')
        rows.append(row); done.add(key); _save_csv(pd.DataFrame(rows), ck)
        if len(done) % 8 == 0 or n == len(tasks) - 1:
            print(f'   {block}: {len(done)}/{len(tasks)}  ({(time.time() - t_start) / 60:.1f} min)  last: {t}  F1={res["f1"]:.4f}')
    print(f'✅ {block} selesai: {len(rows)} baris → {ck}')
    return pd.DataFrame(rows)

def tasks_grid(**axes):
    """Produk kartesius; urutan: dataset → model → run → tuner (sama dengan loop utama)."""
    import itertools
    keys = list(axes)
    return [dict(zip(keys, v)) for v in itertools.product(*axes.values())]

# --- dokumentasi preprocessing untuk R1.6 ---
enc_rows = []
for ds, data in datasets_xy.items():
    pre = clone(data['preprocessor']).fit(data['X'])
    n_out = pre.transform(data['X'].head(5)).shape[1]
    cats = {c: sorted(data['X'][c].dropna().astype(str).unique().tolist()) for c in data['cat_cols']}
    allnan = [c for c in data['num_cols'] if data['X'][c].isna().all()]
    miss = data['X'].isna().mean().sort_values(ascending=False)
    enc_rows.append({'dataset': ds, 'n_rows': len(data['X']), 'n_num': len(data['num_cols']), 'n_cat': len(data['cat_cols']),
                     'cat_levels': json.dumps(cats), 'all_nan_dropped': json.dumps(allnan), 'n_features_after_encoding': n_out,
                     'cols_with_missing': int((miss > 0).sum()), 'max_missing_share': round(float(miss.max()), 3),
                     'onehot': 'OneHotEncoder(handle_unknown=ignore, drop=None) — biner menghasilkan 2 kolom saling komplemen',
                     'target_mapping': json.dumps({str(k): int(v) for k, v in (data['mapping'] or {}).items()})})
enc = pd.DataFrame(enc_rows); _save_csv(enc, os.path.join(REV_DIR, 'preprocessing_summary.csv')); print(enc.T)
miss_tbl = pd.concat({ds: d['X'].isna().mean() for ds, d in datasets_xy.items()}, axis=1).round(3)
_save_csv(miss_tbl.reset_index().rename(columns={'index': 'feature'}), os.path.join(REV_DIR, 'missing_share_per_feature.csv'))

# --- ID baris dan indeks split per run (R1.7) ---
split_rows = []
for ds, data in datasets_split.items():
    X_all, y_all, raw = full_xy_ids(ds, data)
    for run in range(N_RUNS):
        tr, te = split_idx(y_all, run)
        split_rows += [{'dataset': ds, 'run': run + 1, 'seed': SEED + run, 'part': 'train', 'row_id': int(r), 'y': int(y)} for r, y in zip(raw[tr], y_all[tr])]
        split_rows += [{'dataset': ds, 'run': run + 1, 'seed': SEED + run, 'part': 'test', 'row_id': int(r), 'y': int(y)} for r, y in zip(raw[te], y_all[te])]
_save_csv(pd.DataFrame(split_rows), os.path.join(REV_DIR, 'split_indices.csv'))
print('✅ split_indices.csv dan preprocessing_summary.csv disimpan')

RUNTIME_NOTE = ('runtime = waktu dinding (wall-clock) panggilan .fit(): untuk tuner mencakup semua fit CV (5 fold × kandidat), '
                'preprocessing dan SMOTE di dalam pipeline, serta refit akhir; untuk No-Tuning hanya satu fit. '
                'Search memakai n_jobs=-1 (2 vCPU Colab); XGBoost/LightGBM memakai thread default; RF n_jobs=-1.')
print(RUNTIME_NOTE)

## Blok B — Pre-split leak dengan test set asli (R1.1, R2.3)
Preprocessing + SMOTE di-fit pada **seluruh data** (bocor), tetapi model dievaluasi hanya pada **pasien test asli yang sama** dengan protokol bebas bocor. Selisih dengan protokol pre-split lama = efek menilai kasus sintetis; selisih dengan protokol bebas bocor = efek training yang terkontaminasi.

In [ ]:
# ============================================================
# B.1 HITUNGAN ASLI vs SINTETIS UNTUK PROTOKOL PRE-SPLIT LAMA (tanpa melatih model)
# ============================================================
cnt_rows = []
for ds in ['Lung Cancer', 'BreastCancer', 'CervicalCancer']:
    data = datasets_split[ds]
    X_all, y_all, raw = full_xy_ids(ds, data); n = len(y_all)
    pre = clone(data['preprocessor']); Xp = pre.fit_transform(X_all)
    Xr, yr = SMOTE(random_state=SEED, k_neighbors=5).fit_resample(Xp, y_all)
    is_syn = np.arange(len(yr)) >= n                     # imblearn menaruh sampel asli dulu, lalu sintetis
    for run in range(N_RUNS):
        trr, ter = train_test_split(np.arange(len(yr)), test_size=0.2, stratify=yr, random_state=SEED + run)
        for part, ix in [('train', trr), ('test', ter)]:
            cnt_rows.append({'dataset': ds, 'run': run + 1, 'protocol': 'Leak-PreSplit (original)', 'part': part,
                             'n_original': int((~is_syn[ix]).sum()), 'n_synthetic': int(is_syn[ix].sum()),
                             'n_class0': int((yr[ix] == 0).sum()), 'n_class1': int((yr[ix] == 1).sum()),
                             'original_row_ids': ' '.join(map(str, raw[ix[~is_syn[ix]]]))})
        tr, te = split_idx(y_all, run)
        for part, ix, syn in [('train', tr, len(yr) - n), ('test', te, 0)]:
            cnt_rows.append({'dataset': ds, 'run': run + 1, 'protocol': 'Leak-PreSplit-OrigTest (block B)', 'part': part,
                             'n_original': len(ix), 'n_synthetic': int(syn),
                             'n_class0': int((y_all[ix] == 0).sum() + (0 if part == 'test' else (yr[n:] == 0).sum())),
                             'n_class1': int((y_all[ix] == 1).sum() + (0 if part == 'test' else (yr[n:] == 1).sum())),
                             'original_row_ids': ' '.join(map(str, raw[ix]))})
cnt = pd.DataFrame(cnt_rows); _save_csv(cnt, os.path.join(REV_DIR, 'B_sample_counts.csv'))
print(cnt.groupby(['dataset', 'protocol', 'part'])[['n_original', 'n_synthetic', 'n_class0', 'n_class1']].mean().round(1))

In [ ]:
# ============================================================
# B.2 JALANKAN: training terkontaminasi, test = pasien asli yang sama dengan protokol bebas bocor
# ============================================================
_B_cache = {}
def presplit_orig_split(ds, run):
    data = datasets_split[ds]
    if ds not in _B_cache:
        X_all, y_all, raw = full_xy_ids(ds, data); n = len(y_all)
        pre = clone(data['preprocessor']); Xp = pre.fit_transform(X_all)
        Xr, yr = SMOTE(random_state=SEED, k_neighbors=5).fit_resample(Xp, y_all)
        _B_cache[ds] = (Xr, yr, y_all, raw, n)
    Xr, yr, y_all, raw, n = _B_cache[ds]
    tr, te = split_idx(y_all, run)
    tr_all = np.concatenate([tr, np.arange(n, len(yr))])  # train asli + SEMUA sintetis (sebagian berasal dari pasien test)
    return Xr[tr_all], yr[tr_all], Xr[te], y_all[te], raw[te]

def fit_B(t):
    seed = SEED + t['run']
    Xtr, ytr, Xte, yte, ids = presplit_orig_split(t['dataset'], t['run'])
    pipe = SkPipeline([('clf', build_model(t['model'], seed))])
    m, rt, cvb, bp, nc, _ = search(t['tuner'], pipe, Xtr, ytr, seed, PARAM_GRID[t['model']], PARAM_RANDOM[t['model']], PARAM_BAYES[t['model']], N_ITER_TUNING)
    return dict(model=m, runtime=rt, cv_best=cvb, params=bp, n_cand=nc, X_te=Xte, y_te=yte, test_ids=ids, minority=MINORITY[t['dataset']])

TASKS_B = tasks_grid(dataset=['Lung Cancer', 'BreastCancer', 'CervicalCancer'], model=MODEL_NAMES, run=list(range(N_RUNS)), tuner=TUNER_NAMES)
res_B = run_block('B_presplit_origtest', TASKS_B, fit_B)

## Blok D — Cervical tanpa variabel skrining (R2.2)
Hinselmann, Schiller dan Citology adalah hasil pemeriksaan skrining yang dicatat bersamaan dengan biopsi. Blok ini mengulang protokol bebas bocor tanpa ketiganya (prediksi risiko, bukan diagnosis berbantu).

In [ ]:
# ============================================================
# D. CERVICAL TANPA SKRINING
# ============================================================
SCREEN_COLS = ['Hinselmann', 'Schiller', 'Citology']
dC = datasets_split['CervicalCancer']
drop = [c for c in SCREEN_COLS if c in dC['X'].columns]
print('Kolom dihapus:', drop)
Xr_ = dC['X'].drop(columns=drop)
pre_ns, num_ns, cat_ns = build_preprocessor(Xr_)
datasets_split['Cervical-NoScreen'] = {**dC, 'X': Xr_, 'X_train': dC['X_train'].drop(columns=drop), 'X_test': dC['X_test'].drop(columns=drop),
                                       'preprocessor': pre_ns, 'num_cols': num_ns, 'cat_cols': cat_ns}
RAW_ROW['Cervical-NoScreen'] = RAW_ROW['CervicalCancer']; MINORITY['Cervical-NoScreen'] = MINORITY['CervicalCancer']

def fit_leakfree(t, space='orig', scoring='f1_weighted', tuners_space=None, pipe_fn=None):
    ds = t['dataset']; data = datasets_split[ds]; seed = SEED + t['run']
    X_all, y_all, raw = full_xy_ids(ds, data); tr, te = split_idx(y_all, t['run'])
    Xtr, Xte, ytr, yte = X_all.iloc[tr].reset_index(drop=True), X_all.iloc[te].reset_index(drop=True), y_all[tr], y_all[te]
    pipe = pipe_fn(data, t, seed) if pipe_fn else make_pipeline(data['preprocessor'], build_model(t['model'], seed))
    if t['tuner'] == 'No-Tuning':
        G = R = Bz = NI = None
    else:
        G, R, Bz, NI = tuners_space if tuners_space else (PARAM_GRID[t['model']], PARAM_RANDOM[t['model']], PARAM_BAYES[t['model']], N_ITER_TUNING)
    m, rt, cvb, bp, nc, cvr = search(t['tuner'], pipe, Xtr, ytr, seed, G, R, Bz, NI, scoring)
    return dict(model=m, runtime=rt, cv_best=cvb, params=bp, n_cand=nc, X_te=Xte, y_te=yte, test_ids=raw[te], minority=MINORITY[ds], cvr=cvr)

TASKS_D = tasks_grid(dataset=['Cervical-NoScreen'], model=MODEL_NAMES, run=list(range(N_RUNS)), tuner=TUNER_NAMES)
res_D = run_block('D_cervical_noscreen', TASKS_D, fit_leakfree)

## Blok F — Fungsi objektif pada Cervical (R2.4)
Tuning dengan objektif MCC dan balanced accuracy (bukan weighted F1). Output memuat sensitivitas, spesifisitas, PPV dan sensitivitas pada spesifisitas 90%.

In [ ]:
# ============================================================
# F. PERBANDINGAN FUNGSI OBJEKTIF
# ============================================================
SCORERS = {'mcc': make_scorer(matthews_corrcoef), 'bal_acc': 'balanced_accuracy'}
def fit_F(t):
    return fit_leakfree(t, scoring=SCORERS[t['objective']])
TASKS_F = tasks_grid(dataset=['CervicalCancer'], objective=list(SCORERS), model=MODEL_NAMES, run=list(range(N_RUNS)), tuner=['Grid', 'Random'])
res_F = run_block('F_objective_cervical', TASKS_F, fit_F)

## Blok E — Imbalance terkontrol pada Breast (R2.1)
Semua kasus jinak dipertahankan; kasus ganas disampel acak (per run) sampai rasio 5:1, 10:1 dan 15:1. Fitur dan outcome tetap, sehingga hanya tingkat imbalance yang berubah.

In [ ]:
# ============================================================
# E. IMBALANCE TERKONTROL
# ============================================================
IR_LEVELS = [5, 10, 15]
def fit_E(t):
    ds = 'BreastCancer'; data = datasets_split[ds]; seed = SEED + t['run']
    X_all, y_all, raw = full_xy_ids(ds, data)
    maj, mino = np.where(y_all != MINORITY[ds])[0], np.where(y_all == MINORITY[ds])[0]
    rng = np.random.RandomState(1000 * t['ir'] + t['run'])
    keep_min = rng.choice(mino, size=int(round(len(maj) / t['ir'])), replace=False)
    sub = np.sort(np.concatenate([maj, keep_min]))
    Xs, ys, rs = X_all.iloc[sub].reset_index(drop=True), y_all[sub], raw[sub]
    tr, te = train_test_split(np.arange(len(ys)), test_size=0.2, stratify=ys, random_state=seed)
    pipe = make_pipeline(data['preprocessor'], build_model(t['model'], seed))
    m, rt, cvb, bp, nc, _ = search(t['tuner'], pipe, Xs.iloc[tr], ys[tr], seed, PARAM_GRID[t['model']], PARAM_RANDOM[t['model']], PARAM_BAYES[t['model']], N_ITER_TUNING)
    return dict(model=m, runtime=rt, cv_best=cvb, params=bp, n_cand=nc, X_te=Xs.iloc[te], y_te=ys[te], test_ids=rs[te], minority=MINORITY[ds],
                extra={'n_minority_total': len(keep_min), 'n_test_minority': int((ys[te] == MINORITY[ds]).sum())})
TASKS_E = tasks_grid(ir=IR_LEVELS, model=['XGBoost', 'SVM'], run=list(range(N_RUNS)), tuner=TUNER_NAMES)
res_E = run_block('E_controlled_ir_breast', TASKS_E, fit_E)

## Blok G — SMOTENC dan RandomOverSampler pada Lung (R1.6)
Lung adalah satu-satunya dataset dengan fitur kategorikal (GENDER). SMOTE biasa setelah one-hot dapat membuat indikator pecahan; blok ini membandingkan SMOTENC (kategori tetap utuh) dan RandomOverSampler (tanpa sampel sintetis).

In [ ]:
# ============================================================
# G. SAMPLER ALTERNATIF
# ============================================================
def pipe_sampler(data, t, seed):
    num_cols, cat_cols = data['num_cols'], data['cat_cols']
    num_pipe = SkPipeline([('imp', SimpleImputer(strategy='median')), ('scl', StandardScaler())])
    if t['sampler'] == 'smotenc':
        cat_pipe = SkPipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ord', OrdinalEncoder())])
        pre = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)], remainder='drop')
        cat_idx = list(range(len(num_cols), len(num_cols) + len(cat_cols)))
        sampler = SMOTENC(categorical_features=cat_idx, random_state=SEED, k_neighbors=5)
    else:
        pre = copy.deepcopy(data['preprocessor']); sampler = RandomOverSampler(random_state=SEED)
    return ImbPipeline([('pre', pre), ('smote', sampler), ('clf', build_model(t['model'], seed))])
def fit_G(t):
    return fit_leakfree(t, pipe_fn=pipe_sampler)
TASKS_G = tasks_grid(dataset=['Lung Cancer'], sampler=['smotenc', 'ros'], model=MODEL_NAMES, run=list(range(N_RUNS)), tuner=TUNER_NAMES)
res_G = run_block('G_sampler_lung', TASKS_G, fit_G)

## Blok H — TabPFN sebagai model default modern (R2.5)
Aktifkan GPU: Runtime → Change runtime type → T4 GPU (lalu jalankan ulang Bagian 0–5 dan R0). TabPFN v2 (Hollmann et al., Nature 2025) dipakai **tanpa tuning**, pada split dan preprocessing yang sama, dengan dan tanpa SMOTE.

In [ ]:
# ============================================================
# H.0 INSTALASI TabPFN v2
# ============================================================
!pip install -q "tabpfn==2.0.9"
# Jika muncul error otorisasi Hugging Face: buka https://huggingface.co/Prior-Labs/TabPFN-v2-clf,
# setujui lisensinya, lalu jalankan:  from huggingface_hub import login; login()

In [ ]:
# ============================================================
# H.1 JALANKAN TabPFN (tanpa tuning)
# ============================================================
import torch
os.environ['TABPFN_ALLOW_CPU_LARGE_DATASET'] = '1'   # Cervical+SMOTE: >1000 sampel latih; izinkan di CPU
from tabpfn import TabPFNClassifier
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
def pipe_tabpfn(data, t, seed):
    clf = TabPFNClassifier(device=DEVICE, random_state=seed)
    steps = [('pre', copy.deepcopy(data['preprocessor']))]
    if t['variant'] == 'smote': steps.append(('smote', SMOTE(random_state=SEED, k_neighbors=5)))
    steps.append(('clf', clf))
    return ImbPipeline(steps)
def fit_H(t):
    return fit_leakfree({**t, 'tuner': 'No-Tuning'}, pipe_fn=pipe_tabpfn)
TASKS_H = tasks_grid(dataset=['Lung Cancer', 'BreastCancer', 'CervicalCancer'], model=['TabPFN'], variant=['no_resampling', 'smote'], run=list(range(N_RUNS)))
res_H = run_block('H_tabpfn', TASKS_H, fit_H)

## Blok C — Domain bersama dan budget yang disamakan (R1.3)
Ketiga tuner memakai **domain diskret yang sama** (memuat konfigurasi default) dan **jumlah kandidat yang sama (24; SVM 24)**. Grid mengevaluasi sub-kisi 24 titik dari domain itu (termasuk default); random dan Bayesian mengambil 24 kandidat dari seluruh domain. Kolom `default_evaluated` mencatat apakah konfigurasi default ikut dievaluasi.

In [ ]:
# ============================================================
# C. MATCHED DOMAIN & BUDGET
# ============================================================
DOMAIN = {
 'XGBoost': {'learning_rate': [0.01, 0.03, 0.1, 0.3], 'max_depth': [3, 4, 6, 8], 'n_estimators': [50, 100, 200, 300], 'subsample': [0.6, 0.8, 1.0]},
 'RandomForest': {'n_estimators': [50, 100, 200, 300], 'max_depth': [None, 3, 5, 10, 20], 'min_samples_split': [2, 5, 10], 'max_features': ['sqrt', 'log2']},
 'SVM': {'C': [0.01, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0], 'kernel': ['rbf', 'linear'], 'gamma': ['scale', 'auto']},
 'LightGBM': {'learning_rate': [0.01, 0.03, 0.1, 0.3], 'n_estimators': [50, 100, 200, 300], 'max_depth': [-1, 3, 5, 7], 'num_leaves': [15, 31, 63]},
}
GRID_SUB = {
 'XGBoost': {'learning_rate': [0.03, 0.1, 0.3], 'max_depth': [3, 6], 'n_estimators': [100, 300], 'subsample': [0.8, 1.0]},
 'RandomForest': {'n_estimators': [100, 300], 'max_depth': [None, 5, 10], 'min_samples_split': [2, 5], 'max_features': ['sqrt', 'log2']},
 'SVM': {'C': [0.1, 0.3, 1.0, 3.0, 10.0, 30.0], 'kernel': ['rbf', 'linear'], 'gamma': ['scale', 'auto']},
 'LightGBM': {'learning_rate': [0.03, 0.1, 0.3], 'n_estimators': [100, 300], 'max_depth': [-1, 5], 'num_leaves': [15, 31]},
}
DEFAULTS = {'XGBoost': {'learning_rate': 0.3, 'max_depth': 6, 'n_estimators': 100, 'subsample': 1.0},
            'RandomForest': {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'max_features': 'sqrt'},
            'SVM': {'C': 1.0, 'kernel': 'rbf', 'gamma': 'scale'},
            'LightGBM': {'learning_rate': 0.1, 'n_estimators': 100, 'max_depth': -1, 'num_leaves': 31}}
BUDGET_C = 24
for m in DOMAIN:
    for k, v in DEFAULTS[m].items():
        assert v in DOMAIN[m][k] and v in GRID_SUB[m][k], (m, k)
    assert int(np.prod([len(v) for v in GRID_SUB[m].values()])) == BUDGET_C, m
pref = lambda d: {'clf__' + k: v for k, v in d.items()}
RF_NONE = 0   # skopt tidak menerima None; 0 dipetakan kembali ke None di dalam model wrapper
def bayes_space(m):
    out = {}
    for k, v in DOMAIN[m].items():
        vals = [RF_NONE if x is None else x for x in v]
        out['clf__' + k] = Categorical(vals)
    return out
class RFNone(RandomForestClassifier):
    """RandomForest yang memetakan max_depth=0 menjadi None (untuk BayesSearchCV)."""
    def fit(self, X, y, sample_weight=None):
        if self.max_depth == 0: self.max_depth = None
        return super().fit(X, y, sample_weight)
def fit_C(t):
    seed = SEED + t['run']; m = t['model']
    def pfn(data, tt, s):
        clf = RFNone(random_state=s, n_jobs=-1) if (m == 'RandomForest' and tt['tuner'] == 'Bayes') else build_model(m, s)
        return make_pipeline(data['preprocessor'], clf)
    o = fit_leakfree(t, tuners_space=(pref(GRID_SUB[m]), pref(DOMAIN[m]), bayes_space(m), BUDGET_C), pipe_fn=pfn)
    dflt = 0
    if o['cvr'] is not None:
        for p in o['cvr']['params']:
            q = {k.replace('clf__', ''): (None if (k.endswith('max_depth') and m == 'RandomForest' and v == 0) else v) for k, v in p.items()}
            if all(q.get(k) == v for k, v in DEFAULTS[m].items()): dflt = 1
    else:
        dflt = 1
    o['extra'] = {'default_evaluated': dflt}
    return o
TASKS_C = tasks_grid(dataset=list(['Lung Cancer', 'BreastCancer', 'CervicalCancer']), model=MODEL_NAMES, run=list(range(N_RUNS)), tuner=TUNER_NAMES)
res_C = run_block('C_matched_space', TASKS_C, fit_C)

## Blok A — Ulang eksperimen utama dengan log lengkap (R1.7)
Sama persis dengan Bagian 6 S7 (seed, split, search space), tetapi menyimpan hyperparameter terpilih, prediksi per pasien dan ID baris. Hasil F1 harus identik dengan `S4_run_level_results_leakfree.csv`; sel A.2 memeriksanya.

In [ ]:
# ============================================================
# A.1 RERUN DENGAN LOG
# ============================================================
TASKS_A = tasks_grid(dataset=['Lung Cancer', 'BreastCancer', 'CervicalCancer'], model=MODEL_NAMES, run=list(range(N_RUNS)), tuner=TUNER_NAMES)
res_A = run_block('A_main_logged', TASKS_A, fit_leakfree)

In [ ]:
# ============================================================
# A.2 CEK REPRODUKSI TERHADAP HASIL S7 (opsional: unggah S4_run_level_results_leakfree.csv ke /content)
# ============================================================
ref_path = '/content/S4_run_level_results_leakfree.csv'
if os.path.exists(ref_path):
    ref = pd.read_csv(ref_path); a = res_A.copy(); a['Run'] = a['run'] + 1
    mrg = ref.merge(a, left_on=['Dataset', 'Model', 'Tuner', 'Run'], right_on=['dataset', 'model', 'tuner', 'Run'], suffixes=('_s7', '_s8'))
    d = (mrg['f1_s7'] - mrg['f1_s8']).abs()
    print(f'Dibandingkan: {len(mrg)} baris | identik (|Δ|<1e-9): {(d < 1e-9).sum()} | maks |ΔF1| = {d.max():.6f}')
    _save_csv(mrg[['Dataset', 'Model', 'Tuner', 'Run', 'f1_s7', 'f1_s8']], os.path.join(REV_DIR, 'A_reproduction_check.csv'))
else:
    print('Lewati: file referensi S7 tidak ada di /content')

## Z — Arsipkan semua hasil revisi ke ZIP

In [ ]:
# ============================================================
# Z. ZIP SEMUA HASIL REVISI (Drive + unduh)
# ============================================================
import zipfile, glob, shutil, datetime
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
zp = os.path.join(OUTPUT_DIR, f'S8_revision_{stamp}.zip')
files = [f for f in glob.glob(os.path.join(REV_DIR, '*')) if f.endswith(('.csv', '.jsonl', '.txt', '.json'))]
files += [f for f in glob.glob(os.path.join(OUTPUT_DIR, '*')) if f.endswith(('.txt', '.csv'))]
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in files: z.write(f, arcname=os.path.basename(f))
try:
    shutil.copy(zp, REV_DIR)
except Exception as e:
    print('Salin ke Drive gagal:', e)
print(f'✅ {len(files)} file → {zp}')
for f in sorted(files): print('   ', os.path.basename(f))
try:
    from google.colab import files as colab_files; colab_files.download(zp)
except Exception:
    pass